### Setup

In [1]:
import numpy as np
import choix
from scipy.optimize import minimize
import scipy.stats as stats
import matplotlib.pyplot as plt
import random
from matplotlib import colors
import pandas as pd
import seaborn as sns
import pickle
import os
import sys

sys.path.append(os.path.abspath('../../'))
sys.path.append(os.path.abspath('../../../'))
from metrics import compute_acc, compute_weighted_acc
from opt_fair import *
from distribution_utils import safe_kendalltau, to_numpy

In [2]:
!nvidia-smi

Sat Jun 27 07:38:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.163.01             Driver Version: 550.163.01     CUDA Version: 12.5     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          Off |   00000000:17:00.0 Off |                    0 |
| N/A   51C    P0             46W /  300W |       1MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
import os
import torch
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# device = torch.device('cpu')
print(device)

cuda


In [4]:
print(f"Current PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")

Current PyTorch version: 2.4.0a0+f70bd71a48.nv24.06
CUDA available: True
CUDA version: 12.5


In [5]:
with open("../data/FaceAgePC.pickle", 'rb') as handle:
    PC_faceage = pickle.load(handle)    
with open("../data/FaceAgeDF.pickle", 'rb') as handle:
    df_faceage = pickle.load(handle)

In [6]:
df_faceage

,full_path,score,gender
0,nm1442940_rm3965098752_1996-10-3_2006.jpg,10,0.0
1,nm4832920_rm1781768448_2003-8-28_2013.jpg,10,0.0
2,nm0652089_rm860657920_1992-3-10_2002.jpg,10,0.0
3,nm0004917_rm1493730304_1969-5-12_1979.jpg,10,0.0
4,nm1113550_rm1332711936_1996-4-14_2006.jpg,10,0.0
...,...,...,...
9145,475367_1941-08-03_2011.jpg,70,1.0
9146,304085_1919-07-07_1989.jpg,70,1.0
9147,nm0001627_rm4164078592_1927-2-20_1997.jpg,70,1.0
9148,nm0000024_rm1715129344_1904-4-14_1974.jpg,70,1.0


In [7]:
import opt_fair
all_pc_faceage  = opt_fair._pc_without_reviewers(PC_faceage)

size = len(df_faceage)
print(size)

9150


In [8]:
print(len(all_pc_faceage))

250249


In [9]:
gt_scores = to_numpy(df_faceage['score'].tolist())

In [10]:
max_iter=30000

In [11]:
lr = 0.001

In [12]:
tol = 1e-6

In [13]:
betas_1 = [
    -1.0, -0.9, -0.8, -0.7, -0.6, -0.5, -0.4, -0.3, -0.2, -0.1,
     0.0,  0.1,  0.2,  0.3,  0.4,  0.5,  0.6,  0.7,  0.8,  0.9, 1.0
]

betas_2 = [
    0.0, 0.1, 0.2, 0.3, 0.4, 0.5,
    0.6, 0.7, 0.8, 0.9, 1.0
]

### HBTL

In [16]:
import random
from grad_em import *

In [18]:
import time
HBTL = defaultdict(list)
gradem_time = []
for sd in range(1):
    for beta in betas_1:
        start = time.time()
        grad_em = GradientEMWrapper(PC_faceage, lr, sd, device, max_iter=max_iter, init_beta=beta)
        r_est, beta_est = grad_em.run_algorithm()
        end = time.time()
        gradem_time.append(end-start)

        r_est_np = to_numpy(r_est)
        gt_scores = to_numpy(df_faceage['score'].tolist())
        current_tau = safe_kendalltau(r_est_np, gt_scores)
        if current_tau < 0:
            r_est_np = -r_est_np
        grad_acc = compute_acc(df_faceage, 1*r_est_np, device)
        grad_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
        grad_tau = safe_kendalltau(r_est_np, gt_scores)

        HBTL[beta] = [grad_acc, grad_wacc, grad_tau]
        print(HBTL[beta])

 42%|███████████████████████████████▉                                            | 12610/30000 [01:03<01:27, 198.93it/s]


HBTL: Converged at epoch 12610 | Loss: 0.255291
[0.7920348644256592, 0.8734531998634338, 0.5792941189766483]


 42%|███████████████████████████████▉                                            | 12609/30000 [01:02<01:26, 200.49it/s]


HBTL: Converged at epoch 12609 | Loss: 0.255272
[0.7920330762863159, 0.8734517097473145, 0.5792906021279148]


 42%|███████████████████████████████▊                                            | 12573/30000 [01:22<01:54, 152.41it/s]


...

HBTL: Converged at epoch 11817 | Loss: 0.255292
[0.7920330166816711, 0.8734513521194458, 0.5792905849598406]


 40%|██████████████████████████████▍                                             | 12023/30000 [01:55<02:52, 104.40it/s]


HBTL: Converged at epoch 12023 | Loss: 0.255257
[0.7920322418212891, 0.8734515309333801, 0.5792889159675632]


 38%|████████████████████████████▌                                               | 11270/30000 [01:47<02:58, 104.73it/s]


KeyboardInterrupt: 

In [ ]:
print(HBTL)

### PG EM

In [ ]:
import random
from pgem_initialize_beta_1 import EMWrapper

In [ ]:
PGEM = defaultdict(list)
pgem_time = []
for sd in range(1):
    for beta in betas_1:
        start = time.time()
        pg = EMWrapper(PC_faceage, max_iter, device, sd, init_beta=beta)
        r_est, beta_est, ll = pg.run_algorithm()
        end = time.time()
        pgem_time.append(end-start)

        if np.isnan(r_est).any() or np.isnan(beta_est).any() or np.isnan(ll):
            print("Skipping nan")
            continue

        r_est_np = to_numpy(r_est)

        gt_scores = to_numpy(df_faceage['score'].tolist())
        current_tau = safe_kendalltau(r_est_np, gt_scores)
        if current_tau < 0:
            r_est_np = -r_est_np
        pgem_acc = compute_acc(df_faceage, 1*r_est_np, device)
        pgem_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
        pgem_tau = safe_kendalltau(r_est_np, gt_scores)

        PGEM[beta] = [pgem_acc, pgem_wacc, pgem_tau]
        print(PGEM[beta])

In [ ]:
print(PGEM)

### CrowdBT

In [ ]:
crowd_labels = pd.read_csv('data/crowd_labels.csv')
num_reviewers =  crowd_labels['performer'].nunique()

In [ ]:
print(device)
gt_scores = to_numpy(df_faceage['score'].tolist())

In [ ]:
CrowdBT = defaultdict(list)
K = num_reviewers
gt_df = df_faceage
crowdbt_time = []
for seed in range(1):
    try:
        for beta in betas_2:
            start = time.time()
            crowdbt_test = opt_fair.CrowdBT_3_0(data=PC_faceage, penalty=0, device=device, random_seed=seed, init_beta=beta)
            crowdbt_scores, _ = crowdbt_test.alternate_optim(size, K, lr_x=lr, lr_y=lr, tol=tol, iters=max_iter)
            end = time.time()
            crowdbt_time.append(end-start)
            r_est_np = to_numpy(crowdbt_scores)
            gt_scores = to_numpy(df_faceage['score'].tolist())
            current_tau = safe_kendalltau(r_est_np, gt_scores)
            if current_tau < 0:
                r_est_np = -r_est_np
            crowdbt_acc = compute_acc(df_faceage, 1*r_est_np, device)
            crowdbt_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
            crowdbt_tau = safe_kendalltau(r_est_np, gt_scores)
            CrowdBT[beta] = [crowdbt_acc, crowdbt_wacc, crowdbt_tau]
            print(CrowdBT[beta])
    except Exception as e:
        print(e)

In [44]:
print(CrowdBT)

CrowdBT -- Accuracy: 0.7917298078536987 ± 0.0, Weighted Accuracy: 0.8730530738830566 ± 0.0, Kendall's Tau: 0.5786890421159417 ± 0.0


### FactorBT

In [45]:
crowd_labels = pd.read_csv('data/crowd_labels.csv')
num_reviewers =  crowd_labels['performer'].nunique()

In [46]:
print(device)
gt_scores = to_numpy(df_faceage['score'].tolist())

cuda


In [47]:
from collections import defaultdict
from pathlib import Path

def build_pc_dict_for_noisybt(df, df_faceage, worker_col="worker", left_col="left", right_col="right", label_col="label"):
    """
    Builds the dict NoisyBT_3_0 expects: {worker_id: [(left, right, winner), ...]}

    Mirrors the exact item/worker id mapping used to build PC_faceage_spm:
      - worker ids: position of first appearance in df[worker_col].unique()
      - item ids: position of df_faceage['full_path'] (matched via Path(...).name,
        since df['left']/df['right']/df['label'] are full URLs but full_path
        entries are basenames)

    Unlike PC_faceage_spm (which collapses each row to (winner, loser)), this
    keeps the left/right slot assignment, since NoisyBT's bias parameter needs
    to know which item was shown in the "left" position.

    Returns the dict AND the two id mappings, so you can invert them later
    if needed (e.g. to map scores back to filenames).
    """
    unique_performers = list(df[worker_col].unique())
    performer_label_dict = {performer: i for i, performer in enumerate(unique_performers)}

    item_labels = list(df_faceage["full_path"])
    item_label_dict = {item: i for i, item in enumerate(item_labels)}

    pc_dict = defaultdict(list)
    for performer, group in df.groupby(worker_col):
        key = performer_label_dict[performer]
        for _, row in group.iterrows():
            left = Path(row[left_col]).name
            right = Path(row[right_col]).name
            winner = Path(row[label_col]).name

            left_id = item_label_dict[left]
            right_id = item_label_dict[right]
            winner_id = item_label_dict[winner]

            pc_dict[key].append((left_id, right_id, winner_id))

    return dict(pc_dict), performer_label_dict, item_label_dict

In [48]:
def sort_df(df, column_name):
    # Sort by a specific column (replace 'column_name' with your column)
    df_sorted = df.sort_values(by=column_name, ascending=True)  # or ascending=False

    return df_sorted

df = pd.read_csv('data/crowd_labels.csv')
df = df.rename(columns={'performer': 'worker'})

In [49]:
df.head()

,left,right,label,worker
0,https://tlk.s3.yandex.net/annotation_tasks/IMD...,https://tlk.s3.yandex.net/annotation_tasks/IMD...,https://tlk.s3.yandex.net/annotation_tasks/IMD...,0
1,https://tlk.s3.yandex.net/annotation_tasks/IMD...,https://tlk.s3.yandex.net/annotation_tasks/IMD...,https://tlk.s3.yandex.net/annotation_tasks/IMD...,0
2,https://tlk.s3.yandex.net/annotation_tasks/IMD...,https://tlk.s3.yandex.net/annotation_tasks/IMD...,https://tlk.s3.yandex.net/annotation_tasks/IMD...,0
3,https://tlk.s3.yandex.net/annotation_tasks/IMD...,https://tlk.s3.yandex.net/annotation_tasks/IMD...,https://tlk.s3.yandex.net/annotation_tasks/IMD...,1
4,https://tlk.s3.yandex.net/annotation_tasks/IMD...,https://tlk.s3.yandex.net/annotation_tasks/IMD...,https://tlk.s3.yandex.net/annotation_tasks/IMD...,1


In [50]:
gt_df = df_faceage
gt_df.head()

,full_path,score,gender
0,nm1442940_rm3965098752_1996-10-3_2006.jpg,10,0.0
1,nm4832920_rm1781768448_2003-8-28_2013.jpg,10,0.0
2,nm0652089_rm860657920_1992-3-10_2002.jpg,10,0.0
3,nm0004917_rm1493730304_1969-5-12_1979.jpg,10,0.0
4,nm1113550_rm1332711936_1996-4-14_2006.jpg,10,0.0


In [51]:
import time
NoisyBT = defaultdict(list)
K = num_reviewers
gt_df = df_faceage
noisybt_time = []

# Build the (left, right, winner) dict NoisyBT_3_0 needs, from the same
# crowd_labels.csv-derived df used by the original NoisyBradleyTerry.
# `df` must have integer-coded 'worker', 'left', 'right', 'label' columns
# aligned with `size`/num_items and gt_df['score'] (see build_pc_dict.py).
PC_faceage_noisybt, _performer_label_dict, _item_label_dict = build_pc_dict_for_noisybt(df, df_faceage)

for seed in range(1):
    try:
        for beta in betas_2:
            start = time.time()
            noisybt_test = opt_fair.NoisyBT_3_0(data=PC_faceage_noisybt, penalty=0, device=device, random_seed=seed, init_beta=beta)
            noisybt_scores, noisybt_skills, noisybt_biases = noisybt_test.alternate_optim(size, K, lr_x=lr, lr_y=lr, tol=tol, iters=max_iter)
            end = time.time()
            noisybt_time.append(end-start)
            r_est_np = to_numpy(noisybt_scores)
            gt_scores = to_numpy(gt_df['score'].tolist())
            current_tau = safe_kendalltau(r_est_np, gt_scores)
            if current_tau < 0:
                r_est_np = -r_est_np
            noisybt_acc = compute_acc(gt_df, 1*r_est_np, device)
            noisybt_wacc = compute_weighted_acc(gt_df, 1*r_est_np, device)
            noisybt_tau = safe_kendalltau(r_est_np, gt_scores)
            NoisyBT_accs[beta] = [noisybt_acc, noisybt_wacc, noisybt_tau]
    except Exception as e:
        print(e)

 84%|████████████████████████████████████████████████████▏         | 25251/30000 [02:27<00:27, 170.69it/s, loss=6.27e+4]


In [52]:
print(FactorBT)

FactorBT -- Accuracy: 0.7916303873062134 ± 0.0, Weighted Accuracy: 0.8729999661445618 ± 0.0, Kendall's Tau: 0.578491747529307 ± 0.0
